# BART backward-elimination feature selection (Run 14)

Estimator-specific selection: XGBoost-based RFE selects features XGBoost can exploit,
and run 12 showed that set hurt BART. This notebook runs `BartBackwardElimination` —
importances re-measured each iteration from BART's own `variable_inclusion`, six
seed-replicate fits in parallel per iteration (12 of 14 cores) to tame PGBART sampler
noise. Progress streams live to `/tmp/bart_rfe_progress.log` (nbconvert buffers cell
stdout until completion, so the sidecar file is the only live view).

**Selection discipline:** every decision below uses the 2022-2023 validation slice; the
2024+2025 hold-out is read exactly ONCE, in the final cell, for the selected set.
Selection is by validation-curve PEAK, not smallest-within-tolerance (see the run-13
experiment-log entry for why the tolerance rule over-shrinks on flat curves).

In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pymc as pm
import pymc_bart as pmb
import os, sys, time, threading
module_path = os.path.abspath(os.path.join('../../../../../'))
if module_path not in sys.path:
    sys.path.append(module_path)
from data_science_utilities.models.bart.feature_selection.backward_elimination import (
    BartBackwardElimination,
)

RANDOM_SEED = 32
TARGET = 'target_win'
BART_NAN_SENTINEL = -100.0
# 91 is 3x the XGBoost CV peak (32) and above the ~60 usable upper bound, so still a
# generous pre-filter, but skips the ~30-min-per-fit 176/140 iterations that explore
# territory the XGBoost RFE already proved worthless.
START_POOL_SIZE = 91

# Per-fit / per-iteration progress to a sidecar file: headless nbconvert buffers a
# cell's stdout until the cell finishes, so this is the only way to watch live.
_LOG_PATH = '/tmp/bart_rfe_progress.log'
_log_lock = threading.Lock()


def log_progress(message):
    line = f"{time.strftime('%H:%M:%S')} {message}\n"
    with _log_lock:
        with open(_LOG_PATH, 'a') as handle:
            handle.write(line)


START_FEATURES = list(pd.read_csv(
    '../../../../../data/predict_games/model_features_in/rfe_features_kfolds.csv',
    index_col=0,
).loc[START_POOL_SIZE].dropna().values)

MODEL_INPUTS_DF = pd.read_parquet(
    '../../../../../data/predict_games/input_data/schedule_and_weekly.parquet'
).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

train_df = MODEL_INPUTS_DF[MODEL_INPUTS_DF['season'] < 2022]
valid_df = MODEL_INPUTS_DF[
    (MODEL_INPUTS_DF['season'] >= 2022) & (MODEL_INPUTS_DF['season'] < 2024)
]
y_train = train_df[TARGET].to_numpy(dtype=int)
y_valid = valid_df[TARGET].to_numpy(dtype=int)
log_progress(f'START run: {len(START_FEATURES)} starting features')
print(len(START_FEATURES), 'starting features')

In [ ]:
def bart_fit(features, seed):
    """One BART fit: train on <2022, score ROC-AUC on the 2022-2023 validation slice,
    return the chain-averaged variable_inclusion. 2 chains / 2 cores per fit so six
    replicate fits run in parallel (12 of 14 cores, 2 left for the system).

    Selection-grade sampling: draws cut to 500 (full tune kept for PGBART mixing) —
    selection only needs the importance ranking and a validation AUROC, both of which
    are posterior means that converge fast, and 6-replicate averaging absorbs the rest.
    The final hold-out fit keeps full 1000-draw / 4-chain sampling."""
    t0 = time.perf_counter()
    X_train = train_df[features].fillna(BART_NAN_SENTINEL).to_numpy(dtype=float)
    X_valid = valid_df[features].fillna(BART_NAN_SENTINEL).to_numpy(dtype=float)
    assert not np.isnan(X_train).any() and not np.isnan(X_valid).any()

    with pm.Model() as model:
        X_data = pm.Data('X', X_train)
        mu = pmb.BART('mu', X_data, y_train, m=50)
        p = pm.Deterministic('p', pm.math.invprobit(mu))
        pm.Bernoulli('y', p=p, observed=y_train, shape=mu.shape)
        idata = pm.sample(draws=500, tune=1000, chains=2, cores=2,
                          random_seed=seed, progressbar=False)
        pm.set_data({'X': X_valid})
        ppc = pm.sample_posterior_predictive(
            idata, var_names=['p'], random_seed=seed, progressbar=False)

    valid_preds = ppc.posterior_predictive['p'].mean(dim=['chain', 'draw']).to_numpy()
    inclusion = pd.Series(
        idata.sample_stats['variable_inclusion']
        .mean(dim=['chain', 'draw']).to_numpy(),
        index=features,
    )
    score = roc_auc_score(y_valid, valid_preds)
    log_progress(f'  fit done: {len(features)} feat, seed {seed}, '
                 f'{time.perf_counter() - t0:.0f}s, valid auroc {score:.4f}')
    return {'validation_score': score, 'variable_inclusion': inclusion}

In [ ]:
def on_iter(row):
    log_progress(f'=== ITER complete: {row["num_features"]} features, '
                 f'mean validation {row["validation_score"]:.4f} ===')


rfe = BartBackwardElimination(bart_fit, drop_rate=0.2, min_features=10,
                              replicates=6, max_workers=6, base_seed=RANDOM_SEED,
                              on_iteration=on_iter)
history = rfe.run(START_FEATURES)
log_progress('DONE selection loop')
history[['num_features', 'validation_score', 'replicate_scores']]

In [ ]:
plt.plot(history['num_features'], history['validation_score'], marker='o')
plt.gca().invert_xaxis()
plt.axhline(history['validation_score'].max(), color='r', linestyle='--')
plt.xlabel('features')
plt.ylabel('validation ROC-AUC (2022-2023)')
best_features = rfe.get_best_features()
print('validation peak at', len(best_features), 'features,',
      f"score {history['validation_score'].max():.4f}")

In [ ]:
pd.DataFrame({'feature': best_features}).to_csv(
    '../../../../../data/predict_games/model_features_in/bart_rfe_features.csv',
    index=False,
)
sorted(best_features)

In [ ]:
# THE single hold-out read: final 4-chain fit on the selected set, evaluated
# exactly as bart.ipynb does.
holdout_df = MODEL_INPUTS_DF[MODEL_INPUTS_DF['season'] >= 2024].copy()
y_holdout = holdout_df[TARGET].to_numpy(dtype=int)
X_train_full = train_df[best_features].fillna(BART_NAN_SENTINEL).to_numpy(dtype=float)
X_holdout = holdout_df[best_features].fillna(BART_NAN_SENTINEL).to_numpy(dtype=float)
assert not np.isnan(X_train_full).any() and not np.isnan(X_holdout).any()

with pm.Model() as final_model:
    X_data = pm.Data('X', X_train_full)
    mu = pmb.BART('mu', X_data, y_train, m=50)
    p = pm.Deterministic('p', pm.math.invprobit(mu))
    pm.Bernoulli('y', p=p, observed=y_train, shape=mu.shape)
    idata_final = pm.sample(draws=1000, tune=1000, chains=4, cores=4,
                            random_seed=RANDOM_SEED, progressbar=False)
    pm.set_data({'X': X_holdout})
    ppc_final = pm.sample_posterior_predictive(
        idata_final, var_names=['p'], random_seed=RANDOM_SEED, progressbar=False)

post_p = ppc_final.posterior_predictive['p']
holdout_preds = post_p.mean(dim=['chain', 'draw']).to_numpy()
holdout_std = post_p.std(dim=['chain', 'draw']).to_numpy()
print(f'final fit on {len(best_features)} features; holdout {X_holdout.shape}')
print(f'2024+2025 hold-out ROC-AUC:  {roc_auc_score(y_holdout, holdout_preds):.4f}')
print(f'2024+2025 hold-out accuracy: {((holdout_preds > 0.5).astype(int) == y_holdout).mean():.4f}')
print(f'Brier score: {brier_score_loss(y_holdout, holdout_preds):.4f}')
print(f'Log loss:    {log_loss(y_holdout, holdout_preds):.4f}')
width_quartile = pd.qcut(pd.Series(holdout_std), 4,
                         labels=['narrowest', 'q2', 'q3', 'widest'])
stratified = pd.DataFrame({'y': y_holdout, 'p': holdout_preds}).groupby(
    width_quartile, observed=True
).apply(lambda x: brier_score_loss(x['y'], x['p']), include_groups=False)
print('Brier by posterior-width quartile:')
print(stratified.round(4).to_string())